In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn 
import numpy as np
import os
import copy
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from scipy import ndimage

# ==========================================
# 0. PERFORMANCE OPTIMIZATIONS
# ==========================================
cudnn.benchmark = True  
cudnn.enabled = True
# Get the number of available CPU cores
num_cores = os.cpu_count() 

# Force PyTorch to use all of them for math operations 
torch.set_num_threads(num_cores)
#torch.set_num_interop_threads(num_cores)

print(f"🧵 CPU Threads set to: {torch.get_num_threads()}")
# ==========================================
# 1. EXPERIMENT HYPERPARAMETERS
# ==========================================
CFG = {
    "data_dir": "/kaggle/input/datasets/williamwhy1/brats2021preprocessed/preprocessed",
    "fold": 0,                      
    "patch_size": (128, 128, 128),  
    "features": [16, 24, 32, 64, 128], 
    "foreground_prob": 0.7,         # 70% of patches will be forced to have tumor
    "foreground_threshold": 0.01,   # At least 1% of pixels must be tumor
    "region_based": True, # Toggle this to switch modes
    "num_classes": 3 if True else 4, # WT, TC, ET if region-based
    "batch_size": 2,                
    "num_workers": 4,               
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    "lr": 5e-4,
    "epochs": 200,
    "iters_per_epoch": 150,        
    "ema_alpha": 0.999,             
    "poly_power": 0.9,

    # Early Stopping & Subset Validation
    "val_every": 5,                # Only validate every 5 epochs
    "patience": 4,                 # Wait 4 validations before killing run
    "min_delta": 0.001,             # 0.1% improvement required
    "val_percent": 1.0             # Validate on 100% of val set each epoch
}

print(f"Using device: {CFG['device']}")

# ==========================================
# 2. DATALOADER & PREPROCESSING
# ==========================================

class BraTSRandomPatchDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, patient_ids, patch_size, fg_prob=0.7, fg_thresh=0.01, augment=True, region_based=False):
        self.data_dir = data_dir
        self.patient_ids = patient_ids
        self.patch_size = patch_size
        self.fg_prob = fg_prob
        self.fg_thresh = fg_thresh
        self.augment = augment
        self.region_based = region_based

    def __len__(self):
        return len(self.patient_ids)

    def _convert_to_regions(self, mask):
        """Converts integer mask (H, W, D) to overlapping regions (3, H, W, D)"""
        # Label 1: Necrotic, 2: Edema, 3: Enhancing
        wt = (mask > 0).astype(np.float32)                # Whole Tumor (1+2+3)
        tc = np.logical_or(mask == 1, mask == 3).astype(np.float32) # Tumor Core (1+3)
        et = (mask == 3).astype(np.float32)                # Enhancing Tumor (3)
        return np.stack([wt, tc, et], axis=0)

    def __getitem__(self, idx):
        p_id = self.patient_ids[idx]
        with np.load(os.path.join(self.data_dir, f"{p_id}.npz")) as data:
            image = data['data']  
            mask = data['seg']    

        mask[mask == 4] = 3
        c, h, w, d = image.shape
        th, tw, td = self.patch_size

        # --- FOREGROUND BALANCING ---
        force_foreground = np.random.random() < self.fg_prob
        i, j, k = np.random.randint(0, h-th+1), np.random.randint(0, w-tw+1), np.random.randint(0, d-td+1)

        if force_foreground:
            for _ in range(15):
                ti, tj, tk = np.random.randint(0, h-th+1), np.random.randint(0, w-tw+1), np.random.randint(0, d-td+1)
                if np.mean(mask[ti:ti+th, tj:tj+tw, tk:tk+td] > 0) >= self.fg_thresh:
                    i, j, k = ti, tj, tk
                    break

        img_patch = image[:, i:i+th, j:j+tw, k:k+td].copy()
        mask_patch = mask[i:i+th, j:j+tw, k:k+td].copy()

        # --- AUGMENTATION PIPELINE ---
        if self.augment:
            if np.random.random() > 0.15:
                axes = np.random.choice([0, 1, 2], size=2, replace=False)
                angle = np.random.uniform(-30, 30)
                img_patch = ndimage.rotate(img_patch, angle, axes=(axes[0]+1, axes[1]+1), reshape=False, order=1, mode='constant', cval=0)
                mask_patch = ndimage.rotate(mask_patch, angle, axes=axes, reshape=False, order=0, mode='constant', cval=0)

            for axis in [1, 2, 3]:
                if np.random.random() > 0.25:
                    img_patch = np.flip(img_patch, axis=axis)
                    mask_patch = np.flip(mask_patch, axis=axis-1)

            if np.random.random() > 0.2:
                img_patch *= np.random.uniform(0.7, 1.3)

            if np.random.random() > 0.15:
                img_patch += np.random.normal(0, 0.05, img_patch.shape)

        # --- OUTPUT CONVERSION ---
        img_tensor = torch.from_numpy(img_patch.copy()).float()
        
        if self.region_based:
            # Returns (3, H, W, D)
            mask_tensor = torch.from_numpy(self._convert_to_regions(mask_patch)).float()
        else:
            # Returns (H, W, D) for standard CrossEntropy
            mask_tensor = torch.from_numpy(mask_patch.copy()).long()
            
        return img_tensor, mask_tensor

def calculate_brats_dice(preds, masks, region_based=True):
    results = {}
    
    if region_based:
        # preds: (B, 3, H, W, D), masks: (B, 3, H, W, D)
        # Apply threshold to sigmoid outputs
        p_wt, m_wt = (preds[:, 0] > 0.5).float(), masks[:, 0].float()
        p_tc, m_tc = (preds[:, 1] > 0.5).float(), masks[:, 1].float()
        p_et, m_et = (preds[:, 2] > 0.5).float(), masks[:, 2].float()
    else:
        # Standard multiclass logic (using your existing code)
        p_wt, m_wt = (preds > 0).float(), (masks > 0).float()
        p_tc = torch.logical_or(preds == 1, preds == 3).float()
        m_tc = torch.logical_or(masks == 1, masks == 3).float()
        p_et, m_et = (preds == 3).float(), (masks == 3).float()

    for name, (p, m) in [('WT', (p_wt, m_wt)), ('TC', (p_tc, m_tc)), ('ET', (p_et, m_et))]:
        inter = (p * m).sum()
        union = p.sum() + m.sum()
        results[name] = ((2. * inter + 1e-8) / (union + 1e-8)).item()
        
    return results

# ==========================================
# 3. MODEL ARCHITECTURE (ResidualUNet3D remains same)
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv3d(in_c, out_c, kernel_size=3, padding=1, bias=False)
        # Swapped BatchNorm for InstanceNorm
        self.in1 = nn.InstanceNorm3d(out_c, affine=True) 
        
        self.conv2 = nn.Conv3d(out_c, out_c, kernel_size=3, padding=1, bias=False)
        self.in2 = nn.InstanceNorm3d(out_c, affine=True)
        
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_c, out_c, kernel_size=1, bias=False),
                nn.InstanceNorm3d(out_c, affine=True)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.in1(self.conv1(x)), inplace=True)
        x = self.in2(self.conv2(x))
        x += residual
        return F.relu(x, inplace=True)

class ResidualUNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=4, features=[16, 24, 32, 64, 128]):
        super().__init__()
        self.encoder = nn.ModuleList()  # Ensure this is initialized first
        self.decoder = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

        # 1. Build Encoder
        for feat in features:
            self.encoder.append(ResidualBlock(in_channels, feat))
            in_channels = feat

        # 2. Bottleneck
        self.bottleneck = ResidualBlock(features[-1], features[-1] * 2)
        curr_channels = features[-1] * 2

        # 3. Build Decoder & Deep Supervision Heads
        self.ds_heads = nn.ModuleList()
        
        for feat in reversed(features):
            self.decoder.append(nn.ConvTranspose3d(curr_channels, feat, kernel_size=2, stride=2))
            self.decoder.append(ResidualBlock(feat * 2, feat))
            curr_channels = feat
            
            # Add a head for every decoder level except the last one (which is final_conv)
            if feat != features[0]:
                self.ds_heads.append(nn.Conv3d(feat, out_channels, kernel_size=1))

        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        
        # Encoder path
        for down in self.encoder:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        outputs = []
        # Decoder path
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)      # Upsample
            skip = skip_connections[i//2]
            
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat((skip, x), dim=1)
            x = self.decoder[i+1](x)    # Residual Block
            
            # Collect DS outputs (if training)
            # We map the index to the corresponding ds_head
            ds_idx = i // 2
            if self.training and ds_idx < len(self.ds_heads):
                outputs.append(self.ds_heads[ds_idx](x))

        final_out = self.final_conv(x)
        
        if self.training:
            # Return [Final, DS_1, DS_2, DS_3]
            return [final_out] + outputs[::-1]
        return final_out
class DeepSupervisionLoss(nn.Module):
    def __init__(self, base_loss):
        super().__init__()
        self.base_loss = base_loss

    def forward(self, outputs, target):
        # If model is in eval mode, it returns a single Tensor
        if not isinstance(outputs, (list, tuple)):
            return self.base_loss(outputs, target)

        # Calculate weights dynamically based on the number of outputs received
        # This prevents the IndexError
        num_scales = len(outputs)
        weights = [1 / (2**i) for i in range(num_scales)]
        sum_weights = sum(weights)
        norm_weights = [w / sum_weights for w in weights]
        
        total_loss = 0
        for i, output in enumerate(outputs):
            # Downsample the target mask to match the scale of the current DS head
            if output.shape[2:] != target.shape[2:]:
                target_ds = F.interpolate(
                    target, 
                    size=output.shape[2:], 
                    mode='trilinear', 
                    align_corners=False
                )
            else:
                target_ds = target
                
            total_loss += norm_weights[i] * self.base_loss(output, target_ds)
            
        return total_loss
class GeneralizedDiceCELoss(nn.Module):
    def __init__(self, num_classes=4, softmax=True):
        super().__init__()
        self.num_classes = num_classes
        self.softmax = softmax

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target)
        if self.softmax:
            pred = F.softmax(pred, dim=1)
            
        target_one_hot = F.one_hot(target, self.num_classes).permute(0, 4, 1, 2, 3).float()
        dims = (0, 2, 3, 4)
        intersection = torch.sum(pred * target_one_hot, dims)
        addition = torch.sum(pred + target_one_hot, dims)
        weights = 1.0 / (torch.sum(target_one_hot, dims)**2 + 1e-8)
        
        numerator = torch.sum(weights * intersection)
        denominator = torch.sum(weights * addition)
        gdl = 1 - (2. * numerator + 1e-8) / (denominator + 1e-8)
        return ce + gdl
class DiceCELoss(nn.Module):
    def __init__(self, num_classes=4, softmax=True):
        super().__init__()
        self.num_classes = num_classes
        self.softmax = softmax

    def forward(self, pred, target):
        # 1. Cross Entropy Loss
        ce = F.cross_entropy(pred, target)
        
        # 2. Prepare Predictions
        if self.softmax:
            pred = F.softmax(pred, dim=1)
            
        # 3. Convert target to One-Hot: (B, H, W, D) -> (B, C, H, W, D)
        target_one_hot = F.one_hot(target, self.num_classes).permute(0, 4, 1, 2, 3).float()
        
        # 4. Calculate Dice per class
        # We sum over Batch, Height, Width, and Depth (dims 0, 2, 3, 4)
        dims = (0, 2, 3, 4)
        intersection = torch.sum(pred * target_one_hot, dims)
        cardinality = torch.sum(pred + target_one_hot, dims)
        
        dice_score = (2. * intersection + 1e-8) / (cardinality + 1e-8)
        
        # 5. Average Dice across classes and convert to Loss
        dice_loss = 1 - dice_score.mean()
        
        return ce + dice_loss
class RegionDiceCELoss(nn.Module):
    """Loss for WT, TC, ET regions (Multi-label)"""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        # target: (B, 3, H, W, D), pred: (B, 3, H, W, D)
        bce_loss = self.bce(pred, target)
        
        pred = torch.sigmoid(pred)
        dims = (0, 2, 3, 4)
        intersection = torch.sum(pred * target, dims)
        cardinality = torch.sum(pred + target, dims)
        
        dice_score = (2. * intersection + 1e-8) / (cardinality + 1e-8)
        dice_loss = 1 - dice_score.mean()
        
        return bce_loss + dice_loss
# ==========================================
# 5. INITIALIZATION & DATA SPLIT
# ==========================================

all_ids = [f.replace('.npz', '') for f in os.listdir(CFG["data_dir"]) if f.endswith('.npz')]
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

# FIX: Added region_based=CFG["region_based"]
train_loader = DataLoader(
    BraTSRandomPatchDataset(
        CFG["data_dir"], train_ids, CFG["patch_size"],
        fg_prob=CFG["foreground_prob"],
        fg_thresh=CFG["foreground_threshold"],
        augment=True,
        region_based=CFG["region_based"]
    ), 
    batch_size=CFG["batch_size"], 
    shuffle=True, 
    num_workers=CFG["num_workers"],
    pin_memory=True, prefetch_factor=4 # Lowered prefetch to save RAM
)

# FIX: Ensure out_channels matches num_classes
model = ResidualUNet3D(out_channels=CFG["num_classes"]).to(CFG["device"])

if torch.cuda.device_count() > 1:
    print(f"🚀 Multi-GPU active: {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

ema_model = copy.deepcopy(model.module if isinstance(model, nn.DataParallel) else model).to(CFG["device"])
for param in ema_model.parameters(): param.requires_grad = False
#optimizer = torch.optim.SGD(model.parameters(), lr=CFG["lr"], momentum=0.99, weight_decay=3e-5, nesterov=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=CFG["epochs"], power=CFG["poly_power"])
base_criterion = RegionDiceCELoss() if CFG["region_based"] else GeneralizedDiceCELoss()
criterion = DeepSupervisionLoss(base_criterion)
scaler = torch.amp.GradScaler(CFG['device'])

# ==========================================
# 6. TRAINING LOOP
# ==========================================
best_dice = 0.0
patience_counter = 0

for epoch in range(CFG["epochs"]):
    model.train()
    train_loss = 0
    train_iter = iter(train_loader)
    train_pbar = tqdm(range(CFG["iters_per_epoch"]), desc=f"Epoch {epoch} [TRAIN]", leave=True)
    
    for i in train_pbar:
        # ... [Your existing training step: data loading, forward, backward, EMA update] ...
        # (Keeping your original format here)
        try:
            images, masks = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader); images, masks = next(train_iter)
        images, masks = images.to(CFG["device"], non_blocking=True), masks.to(CFG["device"], non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=CFG['device'].type):
            output = model(images); loss = criterion(output, masks)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        with torch.no_grad():
            m = model.module if isinstance(model, nn.DataParallel) else model
            for ema_p, mod_p in zip(ema_model.parameters(), m.parameters()):
                ema_p.data.mul_(CFG["ema_alpha"]).add_(mod_p.data, alpha=1.0 - CFG["ema_alpha"])
        train_loss += loss.item()
        train_pbar.set_postfix({"loss": f"{train_loss/(i+1):.4f}"})

    # --- CONDITIONAL VALIDATION ---
    # Triggered only every X epochs
    if (epoch + 1) % CFG["val_every"] == 0:
        ema_model.eval()
        val_metrics = {'WT': [], 'TC': [], 'ET': []}
        
        num_val = int(len(val_ids) * CFG["val_percent"])
        epoch_val_ids = np.random.choice(val_ids, max(1, num_val), replace=False)
        
        sub_val_loader = DataLoader(
            BraTSRandomPatchDataset(CFG["data_dir"], epoch_val_ids, CFG["patch_size"], augment=False,region_based=CFG["region_based"]),
            batch_size=1, num_workers=CFG["num_workers"], 
            pin_memory=True, prefetch_factor=16
        )
        
        val_pbar = tqdm(sub_val_loader, desc=f"Epoch {epoch} [VAL CHECK]", leave=True)
        with torch.no_grad():
            for v_img, v_mask in val_pbar:
                v_img, v_mask = v_img.to(CFG["device"]), v_mask.to(CFG["device"])
                with torch.amp.autocast(device_type=CFG['device'].type):
                    v_out = ema_model(v_img)
                    # If region based, v_out is already (1, 3, H, W, D). Use Sigmoid.
                    if CFG["region_based"]:
                        v_out_final = torch.sigmoid(v_out)
                    else:
                        v_out_final = torch.argmax(v_out, dim=1)
                
                res = calculate_brats_dice(v_out_final, v_mask, region_based=CFG["region_based"])
                for k in val_metrics: val_metrics[k].append(res[k])
                
                # Updated Progress Bar to show all classes
                val_pbar.set_postfix({
                    "WT": f"{np.mean(val_metrics['WT']):.3f}",
                    "TC": f"{np.mean(val_metrics['TC']):.3f}",
                    "ET": f"{np.mean(val_metrics['ET']):.3f}"
                })

        avg_wt = np.mean(val_metrics['WT'])
        avg_tc = np.mean(val_metrics['TC'])
        avg_et = np.mean(val_metrics['ET'])
        mean_dice = (avg_wt + avg_tc + avg_et) / 3

        # Updated Print Statement for full clinical visibility
        print(f"📊 [VAL CHECK] EMA Dice -> WT: {avg_wt:.4f} | TC: {avg_tc:.4f} | ET: {avg_et:.4f} | Mean: {mean_dice:.4f}")

        # --- EARLY STOPPING ---
        if mean_dice > (best_dice + CFG["min_delta"]):
            best_dice = mean_dice
            patience_counter = 0
            torch.save(ema_model.state_dict(), f"best_model_fold{CFG['fold']}.pth")
            print(f"⭐ New Best Mean Dice! Saved EMA Model.")
        else:
            patience_counter += 1
            print(f"⚠️ Patience: {patience_counter}/{CFG['patience']} validation checks")

        if patience_counter >= CFG["patience"]:
            print(f"🛑 Early stopping triggered. Model hasn't improved for {CFG['patience']} validation events.")
            break
    else:
        # Just a small status update for non-validation epochs
        print(f"🏃 Epoch {epoch} complete (Training only) loss: {train_loss/(i+1):.4f}")

    torch.save(ema_model.state_dict(), f"model_fold{CFG['fold']}_latest.pth")
    scheduler.step()

🧵 CPU Threads set to: 4
Using device: cuda


Epoch 0 [TRAIN]:   0%|          | 0/150 [00:00<?, ?it/s]

🏃 Epoch 0 complete (Training only) loss: 1.2824


Epoch 1 [TRAIN]:   0%|          | 0/150 [00:00<?, ?it/s]

🏃 Epoch 1 complete (Training only) loss: 0.9052


...

Epoch 83 [TRAIN]:   0%|          | 0/150 [00:00<?, ?it/s]

🏃 Epoch 83 complete (Training only) loss: 0.1443


Epoch 84 [TRAIN]:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 84 [VAL CHECK]:   0%|          | 0/251 [00:00<?, ?it/s]

📊 [VAL CHECK] EMA Dice -> WT: 0.9015 | TC: 0.8759 | ET: 0.8362 | Mean: 0.8712
⚠️ Patience: 4/4 validation checks
🛑 Early stopping triggered. Model hasn't improved for 4 validation events.
